In [1]:
import pandas as pd
import re
from collections import Counter
from unidecode import unidecode

df = pd.read_csv('data/scp_03.csv')
df['id_vaga'] = df.index

# ==========================================
# NORMALIZAÇÕES
# ==========================================

def clean_text_search(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'\bc\+\+\b', 'cplusplus', text, flags=re.I)
    text = re.sub(r'\bc#\b', 'csharp', text, flags=re.I)

    text = unidecode(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s#+.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text


def normalize_col_name(text):
    if not isinstance(text, str):
        return None

    t = text.lower().strip()

    # exceções semânticas
    replacements = {
        'c#': 'csharp',
        'c++': 'cplusplus',
        '.net': 'dotnet',
        'node.js': 'nodejs'
    }

    for k, v in replacements.items():
        t = t.replace(k, v)

    # normalização padrão
    t = unidecode(t)
    t = re.sub(r'[^a-z0-9]+', '_', t)
    t = re.sub(r'_+', '_', t).strip('_')

    return f"sk_{t}"


# ==========================================
# VALIDAÇÕES BÁSICAS
# ==========================================

required_cols = ['area', 'nivel', 'Cargo', 'id_vaga']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Colunas ausentes no df: {missing}")

for c in ['area', 'nivel', 'Cargo']:
    df[c] = df[c].astype(str).str.strip().str.title()


# ==========================================
# 1. PREPARAÇÃO DOS METADADOS
# ==========================================

df_crit = pd.read_excel('data/criterios.xlsx', sheet_name='skills')

skill_meta = []
for _, row in df_crit.iterrows():
    c_name = normalize_col_name(row['Skill'])
    skill_meta.append({
        'col_name': c_name,
        'Skill Real': row['Skill'],
        'Tipo': str(row['Tipo']).strip().lower(),
        'Classe': str(row['Classe']).strip(),
        'Domínio': str(row['Domínio']).strip().lower()
    })

df_meta = pd.DataFrame(skill_meta)

cols_existentes = [c for c in df_meta['col_name'] if c in df.columns]
df_meta = df_meta[df_meta['col_name'].isin(cols_existentes)]


# ==========================================
# FUNÇÃO DE RANKING
# ==========================================

def get_ranking(df_target, meta_filter=None, group_col=None, top_n=None):

    local_meta = df_meta.copy()

    if meta_filter:
        for k, v in meta_filter.items():
            local_meta = local_meta[local_meta[k] == v]

    cols_to_sum = local_meta['col_name'].tolist()

    if not cols_to_sum:
        return pd.DataFrame()

    if group_col:
        res = df_target.groupby(group_col)[cols_to_sum].sum().reset_index()
        res = res.melt(id_vars=group_col, var_name='col_name', value_name='Qtd')
    else:
        res = df_target[cols_to_sum].sum().reset_index()
        res.columns = ['col_name', 'Qtd']

    res = res.merge(local_meta[['col_name', 'Skill Real', 'Classe', 'Domínio']], on='col_name')

    if group_col:
        res = res.sort_values([group_col, 'Qtd'], ascending=[True, False])
        if top_n:
            res = res.groupby(group_col).head(top_n)
    else:
        res = res.sort_values('Qtd', ascending=False)
        if top_n:
            res = res.head(top_n)

    return res

def get_ranking_percentual(df_target, meta_filter=None, group_col='area', top_n=5):

    local_meta = df_meta.copy()

    if meta_filter:
        for k, v in meta_filter.items():
            local_meta = local_meta[local_meta[k] == v]

    cols_to_use = local_meta['col_name'].tolist()

    if not cols_to_use:
        return pd.DataFrame()

    # total por grupo
    total_por_grupo = df_target.groupby(group_col).size()

    # soma das skills
    grouped = df_target.groupby(group_col)[cols_to_use].sum()

    # percentual
    perc = grouped.div(total_por_grupo, axis=0) * 100

    # formato longo
    long = (
        perc.reset_index()
        .melt(id_vars=group_col, var_name='col_name', value_name='Perc')
        .merge(local_meta[['col_name', 'Skill Real']], on='col_name')
    )

    # ranking por grupo
    long['Rank'] = (
        long.groupby(group_col)['Perc']
        .rank(method='dense', ascending=False)
        .astype(int)
    )

    # top n
    long = long.sort_values([group_col, 'Rank'])
    long = long.groupby(group_col).head(top_n)

    # formato wide
    wide = long.pivot(index='Skill Real', columns=group_col, values=['Perc', 'Rank'])

    # montar texto final
    final = pd.DataFrame(index=wide.index)

    for area in wide['Perc'].columns:
        p = wide['Perc'][area]
        r = wide['Rank'][area]

        final[area] = p.round(2).astype(str) + '% (' + r.astype(str) + 'º)'

    # ordenar por média geral
    media = wide['Perc'].mean(axis=1)
    final = final.loc[media.sort_values(ascending=False).index]

    return final


def get_ranking_percentual_exec(df_target, meta_filter=None, group_col="area", top_n=5):

    local_meta = df_meta.copy()

    # filtrar metadados
    if meta_filter:
        for k, v in meta_filter.items():
            local_meta = local_meta[local_meta[k] == v]

    cols_to_use = local_meta["col_name"].tolist()

    if not cols_to_use:
        return pd.DataFrame()

    # =========================
    # TOTAL POR ÁREA
    # =========================
    total_por_grupo = df_target.groupby(group_col).size()

    # =========================
    # SOMA DAS SKILLS
    # =========================
    grouped = df_target.groupby(group_col)[cols_to_use].sum()

    # =========================
    # PERCENTUAL
    # =========================
    perc = grouped.div(total_por_grupo, axis=0) * 100

    # =========================
    # FORMATO LONGO
    # =========================
    long = (
        perc.reset_index()
        .melt(id_vars=group_col, var_name="col_name", value_name="Perc")
        .merge(local_meta[["col_name", "Skill Real"]], on="col_name")
    )

    # =========================
    # RANK POR ÁREA
    # =========================
    long["Rank"] = (
        long.groupby(group_col)["Perc"]
        .rank(method="dense", ascending=False)
    )

    # manter só top N por área
    long = long[long["Rank"] <= top_n]

    # =========================
    # CRIAR TABELA BASE (sem NaN)
    # =========================
    skills = long["Skill Real"].unique()
    areas = df_target[group_col].unique()

    final = pd.DataFrame(index=skills, columns=areas)

    # preencher manualmente (evita bug de pivot)
    for _, row in long.iterrows():
        final.loc[row["Skill Real"], row[group_col]] = (
            f"{row['Perc']:.1f}% ({int(row['Rank'])}º)"
        )

    # preencher faltantes com —
    final = final.fillna("—")

    # =========================
    # ORDENAR POR RELEVÂNCIA GLOBAL
    # =========================
    media = (
        long.groupby("Skill Real")["Perc"]
        .mean()
        .sort_values(ascending=False)
    )

    final = final.loc[media.index]

    return final

print("Funções carregadas.")

/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13900/1368612803.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['id_vaga'] = df.index


Funções carregadas.


In [2]:
cols = [
    "sk_comunicacao",
    "sk_criatividade",
    "sk_organizacao",
    "sk_trabalho_em_equipe",
    "sk_resolucao_de_problemas",
    "sk_lideranca",
    "sk_proatividade",
    "sk_pensamento_analitico",
    "sk_autonomia",
    "sk_adaptabilidade"
]

check = df.groupby('area')[cols].sum()

display(check)

,sk_comunicacao,sk_criatividade,sk_organizacao,sk_trabalho_em_equipe,sk_resolucao_de_problemas,sk_lideranca,sk_proatividade,sk_pensamento_analitico,sk_autonomia,sk_adaptabilidade
area,,,,,,,,,,
Automação De Processos E Sistemas,113,90,80,32,33,26,19,24,23,9
Dados E Inteligência Artificial,1100,831,865,414,320,495,261,656,342,153
Desenvolvimento De Software,1817,1441,1482,795,642,734,410,227,558,252
Infraestrutura E Suporte,1372,756,1179,500,612,431,403,130,255,163
Projetos E Gerenciamento,1526,1145,1214,463,468,702,277,209,304,196
Segurança Da Informação,155,109,143,79,28,68,51,24,45,25
Testes E Controle De Qualidade,377,299,380,182,146,178,118,101,76,62



### 2. EXECUÇÃO DAS ANÁLISES


In [3]:
df_meta = df_meta.copy()

df_meta['Domínio'] = (
    df_meta['Domínio']
    .astype(str)
    .str.strip()
)

df_meta.loc[df_meta['Domínio'].isin(['nan', 'NaN', 'None', '']), 'Domínio'] = pd.NA


In [4]:
# 1. Soft Skill Ranking Geral
df_soft_geral = get_ranking(df, meta_filter={'Tipo': 'soft skill'})
df_soft_geral['Porcentagem'] = df_soft_geral['Qtd'] * 100 / 19293
df_soft_geral['Porcentagem'] = df_soft_geral['Porcentagem'].round(2)
df_soft_geral['Porcentagem'] = df_soft_geral['Porcentagem'].astype(str) + '%'
display(df_soft_geral.head(10))

,col_name,Qtd,Skill Real,Classe,Domínio,Porcentagem
2,sk_comunicacao,6460,Comunicação,nan,NaN,33.48%
11,sk_organizacao,5343,Organização,nan,NaN,27.69%
3,sk_criatividade,4671,Criatividade,nan,NaN,24.21%
9,sk_lideranca,2634,Liderança,nan,NaN,13.65%
17,sk_trabalho_em_equipe,2465,Trabalho em equipe,nan,NaN,12.78%
16,sk_resolucao_de_problemas,2249,Resolução de problemas,nan,NaN,11.66%
1,sk_autonomia,1603,Autonomia,nan,NaN,8.31%
14,sk_proatividade,1539,Proatividade,nan,NaN,7.98%
12,sk_pensamento_analitico,1371,Pensamento analítico,nan,NaN,7.11%
0,sk_adaptabilidade,860,Adaptabilidade,nan,NaN,4.46%


In [5]:
# 2. Soft Skill por Área
df_soft_area = get_ranking_percentual_exec(df, meta_filter={'Tipo': 'soft skill'}, group_col='area', top_n=7)
display(df_soft_area)

,Desenvolvimento De Software,Projetos E Gerenciamento,Dados E Inteligência Artificial,Infraestrutura E Suporte,Testes E Controle De Qualidade,Segurança Da Informação,Automação De Processos E Sistemas
Skill Real,,,,,,,
Comunicação,38.2% (1º),45.6% (1º),44.8% (1º),43.0% (1º),34.0% (2º),41.2% (1º),40.4% (1º)
Organização,31.1% (2º),36.2% (2º),35.2% (2º),37.0% (2º),34.3% (1º),38.0% (2º),28.6% (3º)
Criatividade,30.3% (3º),34.2% (3º),33.9% (3º),23.7% (3º),27.0% (3º),29.0% (3º),32.1% (2º)
Pensamento analítico,—,—,26.7% (4º),—,—,—,8.6% (7º)
Liderança,15.4% (5º),21.0% (4º),20.2% (5º),13.5% (6º),16.1% (5º),18.1% (5º),9.3% (6º)
Trabalho em equipe,16.7% (4º),13.8% (6º),16.9% (6º),15.7% (5º),16.4% (4º),21.0% (4º),11.4% (5º)
Resolução de problemas,13.5% (6º),14.0% (5º),—,19.2% (4º),13.2% (6º),—,11.8% (4º)
Proatividade,—,—,—,12.6% (7º),10.6% (7º),13.6% (6º),—
Autonomia,11.7% (7º),9.1% (7º),13.9% (7º),—,—,12.0% (7º),—


In [6]:
print("\n--- 3. Soft Skill x Senioridade (Top 5 por nível) ---")

df_soft_senior = get_ranking_percentual_exec(
    df,
    meta_filter={'Tipo': 'soft skill'},
    group_col='nivel',
    top_n=7
)

# remove coluna N/F se existir
df_soft_senior = df_soft_senior.drop(columns=['N/F'], errors='ignore')

display(df_soft_senior)


--- 3. Soft Skill x Senioridade (Top 5 por nível) ---


,Nível Liderança,Nível Júnior,Nível Pleno / Sênior
Skill Real,,,
Comunicação,52.1% (1º),36.9% (2º),40.9% (1º)
Organização,39.7% (3º),40.3% (1º),32.5% (3º)
Criatividade,30.5% (4º),28.4% (3º),33.5% (2º)
Liderança,51.6% (2º),—,14.2% (6º)
Trabalho em equipe,14.9% (5º),15.3% (4º),18.3% (4º)
Resolução de problemas,14.9% (5º),12.8% (6º),15.4% (5º)
Proatividade,—,13.2% (5º),—
Autonomia,10.9% (6º),—,13.9% (7º)
Negociação,10.3% (7º),—,—


In [7]:
# 4. Ranking de Domínio (por vaga)
df_melted = df.melt(id_vars=['id_vaga'], value_vars=df_meta['col_name'], var_name='col_name', value_name='active')
df_melted = df_melted[df_melted['active'] == 1]
df_melted = df_melted.merge(df_meta[['col_name', 'Domínio']], on='col_name')

df_dominio = df_melted.groupby('Domínio')['id_vaga'].nunique().reset_index(name='Total_Vagas')
display(df_dominio.sort_values('Total_Vagas', ascending=False).head(10))

,Domínio,Total_Vagas
5,cloud computing,3918
3,banco de dados,3756
13,devops e versionamento,3214
0,agilidade e processos,3213
2,backend,2860
9,dados e bi,2645
7,dados,2393
20,integração / apis,2288
16,frontend,1990
33,sistemas operacionais,1521


In [8]:
# ==========================================
# 5. Domínio por Área — formato executivo (sem NaN)
# ==========================================

TOP_N = 9

res_dom_area = []
domains = df_meta['Domínio'].dropna().unique()

# remove áreas NaN do total
total_por_area = df[df['area'].notna()].groupby('area').size()

for dom in domains:
    cols_dom = df_meta[df_meta['Domínio'] == dom]['col_name'].tolist()
    if not cols_dom:
        continue

    # trata NaN como 0 antes da soma
    base = df[cols_dom].fillna(0)

    # presença real
    presenca = (base.sum(axis=1) > 0)

    soma = (
        presenca[df['area'].notna()]
        .groupby(df['area'])
        .sum()
        .reset_index(name='Qtd')
    )

    soma['Domínio'] = dom
    res_dom_area.append(soma)

df_dom_area = pd.concat(res_dom_area, ignore_index=True)

# percentual seguro
df_dom_area['Perc'] = df_dom_area.apply(
    lambda x: (x['Qtd'] / total_por_area.get(x['area'], 1)) * 100,
    axis=1
)

# ranking por área
df_dom_area['Rank'] = (
    df_dom_area.groupby('area')['Perc']
    .rank(method='dense', ascending=False)
    .astype(int)
)

# top N
df_dom_area = df_dom_area.sort_values(['area', 'Rank'])
df_dom_area = df_dom_area.groupby('area').head(TOP_N)

# texto visual
df_dom_area['Texto'] = (
    df_dom_area['Domínio']
    + ' ('
    + df_dom_area['Perc'].round(1).astype(str)
    + '%)'
)

# pivot final
pivot = df_dom_area.pivot_table(index='Rank', columns='area', values='Texto', aggfunc='first').sort_index()

print("\nDomínio por Área — Presença (sem NaN)")
display(pivot)



Domínio por Área — Presença (sem NaN)


area,Automação De Processos E Sistemas,Dados E Inteligência Artificial,Desenvolvimento De Software,Infraestrutura E Suporte,Projetos E Gerenciamento,Segurança Da Informação,Testes E Controle De Qualidade
Rank,,,,,,,
1,dados e bi (20.4%),banco de dados (51.3%),backend (38.1%),sistemas operacionais (27.9%),agilidade e processos (24.3%),cloud computing (49.2%),agilidade e processos (22.5%)
2,dados (16.8%),dados (47.4%),devops e versionamento (37.0%),cloud computing (27.1%),erp (17.6%),segurança (42.6%),dados e bi (21.1%)
3,agilidade e processos (13.2%),dados e bi (40.0%),banco de dados (33.2%),agilidade e processos (14.6%),dados e bi (16.9%),sistemas operacionais (19.9%),produtividade (18.6%)
4,banco de dados (11.4%),cloud computing (38.1%),frontend (30.3%),devops e versionamento (13.9%),cloud computing (11.8%),segurança e compliance (19.1%),devops e versionamento (15.7%)
5,produtividade (10.0%),inteligência artificial (25.1%),integração / apis (30.1%),produtividade (13.7%),backend (10.6%),devops e versionamento (18.4%),qa / testes (11.7%)
6,integração / apis (8.2%),devops e versionamento (23.0%),cloud computing (29.6%),banco de dados (13.0%),banco de dados (10.1%),dados (17.6%),backend (11.2%)
7,backend (6.4%),agilidade e processos (17.8%),agilidade e processos (24.7%),dados e bi (10.9%),produtividade (9.4%),integração / apis (10.4%),integração / apis (11.1%)
8,frontend (5.7%),engenharia de dados (17.2%),arquitetura / backend (17.9%),observabilidade (9.0%),gestão de produto (6.8%),agilidade e processos (9.8%),cloud computing (10.6%)
9,cloud computing (5.0%),plataformas de dados (15.7%),dados (13.5%),backend (8.3%),integração / apis (6.3%),observabilidade (7.4%),banco de dados (9.9%)


In [9]:
# ==========================================
# 6–9. Rankings por Classe — formato executivo
# ==========================================
classes_analise = ['Sistema', 'Ferramenta', 'Linguagem', 'Conceito']

TOP_N = 15
total_vagas = len(df)

for cls in classes_analise:
    print(f"\nRanking de {cls}")

    meta_cls = df_meta[df_meta['Classe'].fillna('').str.contains(cls, case=False)]
    cols_cls = meta_cls['col_name'].tolist()

    if not cols_cls:
        continue

    soma = df[cols_cls].sum().sort_values(ascending=False).head(TOP_N)

    res = pd.DataFrame({
        'col_name': soma.index,
        'Qtd': soma.values
    }).merge(meta_cls[['col_name', 'Skill Real']], on='col_name')

    res['Perc'] = (res['Qtd'] / total_vagas) * 100
    res['Rank'] = range(1, len(res) + 1)

    # texto visual
    res['Texto'] = (
        res['Skill Real']
        + ' ('
        + res['Perc'].round(1).astype(str)
        + '% | '
        + res['Qtd'].astype(int).astype(str)
        + ')'
    )

    # formato final
    pivot = res.set_index('Rank')[['Texto']]

    display(pivot)



Ranking de Sistema


,Texto
Rank,
1,AWS (13.8% | 2137)
2,Azure (10.5% | 1628)
3,SAP (7.0% | 1084)
4,Kubernetes (6.6% | 1017)
5,Windows (6.5% | 1004)
6,Linux (6.2% | 960)
7,GCP (6.0% | 933)
8,React (5.6% | 868)
9,PostgreSQL (5.2% | 814)



Ranking de Ferramenta


,Texto
Rank,
1,Excel (13.1% | 2033)
2,Git (10.6% | 1645)
3,Office 365 (9.2% | 1428)
4,Power BI (8.1% | 1252)
5,Docker (7.5% | 1158)
6,Grafana (6.0% | 925)
7,Gestão de tarefas (3.8% | 593)
8,Terraform (3.6% | 553)
9,Tableau (2.0% | 312)



Ranking de Linguagem


,Texto
Rank,
1,SQL (18.2% | 2825)
2,Python (15.4% | 2393)
3,Javascript (8.9% | 1377)
4,Java (7.9% | 1225)
5,Go (6.1% | 939)
6,Typescript (4.1% | 632)
7,C# (4.0% | 628)
8,HTML (4.0% | 619)
9,CSS (3.8% | 588)



Ranking de Conceito


,Texto
Rank,
1,Cloud (18.5% | 2877)
2,Metodologias Ágeis (17.4% | 2704)
3,API (14.5% | 2256)
4,CI/CD (11.7% | 1812)
5,Scrum (7.7% | 1194)
6,Microservices (5.6% | 872)
7,NoSQL (5.5% | 856)
8,Kanban (5.2% | 810)
9,Machine Learning (5.2% | 802)


In [10]:
# ==========================================
# 10. Top Classe por Área — formato ranking visual
# ==========================================

TOP_N = 10

for cls in classes_analise:
    print(f"\nTop {cls} por Área")

    meta_cls = df_meta[df_meta['Classe'].fillna('').str.contains(cls, case=False)]
    cols_cls = meta_cls['col_name'].tolist()

    if not cols_cls:
        continue

    total_por_area = df.groupby('area').size()

    res = df.groupby('area')[cols_cls].sum().reset_index()
    res = res.melt(id_vars='area', var_name='col_name', value_name='Qtd')
    res = res.merge(meta_cls[['col_name', 'Skill Real']], on='col_name')

    # percentual
    res['Perc'] = res.apply(
        lambda x: (x['Qtd'] / total_por_area[x['area']]) * 100,
        axis=1
    )

    # ranking por área
    res['Rank'] = (
        res.groupby('area')['Perc']
        .rank(method='dense', ascending=False)
        .astype(int)
    )

    # top N
    res = res.sort_values(['area', 'Rank'])
    res = res.groupby('area').head(TOP_N)

    # texto visual
    res['Texto'] = res['Skill Real'] + ' (' + res['Perc'].round(1).astype(str) + '%)'

    # pivot: linha = rank | coluna = área
    pivot = res.pivot_table(
    index='Rank',
    columns='area',
    values='Texto',
    aggfunc='first'   # pega o primeiro se houver empate
    )
    pivot = pivot.sort_index()

    display(pivot)



Top Sistema por Área


area,Automação De Processos E Sistemas,Dados E Inteligência Artificial,Desenvolvimento De Software,Infraestrutura E Suporte,Projetos E Gerenciamento,Segurança Da Informação,Testes E Controle De Qualidade
Rank,,,,,,,
1,SAP (3.9%),AWS (20.8%),AWS (19.0%),Windows (22.5%),SAP (17.6%),Splunk (29.8%),SAP (6.0%)
2,Windows (2.5%),Azure (18.4%),React (14.9%),Linux (16.0%),AWS (3.8%),AWS (28.5%),Azure (5.5%)
3,AWS (2.1%),Snowflake (15.7%),PostgreSQL (11.8%),AWS (13.9%),Salesforce (3.3%),Azure (26.3%),AWS (3.6%)
4,Power Apps (1.8%),GCP (12.7%),Kafka (11.1%),Azure (12.2%),Azure (3.2%),GCP (20.2%),Linux (3.2%)
5,Snowflake (1.1%),Spark (12.5%),Azure (10.8%),Kubernetes (8.7%),Windows (2.0%),Linux (18.9%),GitHub (3.1%)
6,NaN,SAP (5.9%),Kubernetes (10.8%),GCP (5.5%),React (1.9%),Windows (16.8%),GitLab (3.0%)
7,NaN,PostgreSQL (4.8%),Angular (9.6%),Oracle (3.6%),Oracle (1.7%),Kubernetes (9.6%),GCP (1.7%)
8,NaN,Oracle (4.6%),Spring Boot (8.1%),GitHub (3.3%),Linux (1.5%),GitHub (3.7%),Oracle (1.6%)
9,NaN,NGINX (4.4%),MySQL (7.2%),SAP (3.0%),GCP (1.5%),SAP (3.2%),NaN



Top Ferramenta por Área


area,Automação De Processos E Sistemas,Dados E Inteligência Artificial,Desenvolvimento De Software,Infraestrutura E Suporte,Projetos E Gerenciamento,Segurança Da Informação,Testes E Controle De Qualidade
Rank,,,,,,,
1,Excel (17.9%),Power BI (30.8%),Git (22.9%),Office 365 (13.7%),Excel (14.5%),Terraform (9.3%),Excel (19.2%)
2,Office 365 (10.0%),Excel (24.1%),Docker (15.2%),Excel (9.1%),Office 365 (9.4%),Grafana (7.4%),Office 365 (18.6%)
3,Power BI (6.4%),Git (12.2%),Excel (8.1%),Grafana (9.0%),Power BI (6.1%),Office 365 (5.3%),Gestão de tarefas (8.7%)
4,Selenium (3.2%),Tableau (9.1%),Office 365 (6.0%),Terraform (7.4%),Gestão de tarefas (5.1%),Docker (4.8%),Cypress (7.9%)
5,Grafana (2.5%),Airflow (7.2%),Grafana (5.9%),Docker (6.0%),Grafana (3.1%),Excel (4.5%),Postman (7.6%)
6,Tableau (1.8%),LangChain (6.3%),Gestão de tarefas (3.2%),Ansible (4.1%),Git (2.0%),Ansible (3.5%),Selenium (7.1%)
7,Gestão de tarefas (1.4%),Grafana (6.2%),Figma (3.0%),Gestão de tarefas (4.1%),Figma (1.9%),Burp Suite (2.7%),Playwright (6.8%)
8,NaN,Docker (6.0%),Terraform (2.8%),Git (3.5%),Tableau (1.3%),Git (2.4%),Power BI (5.8%)
9,NaN,Office 365 (5.5%),Power BI (2.4%),Power BI (2.9%),Docker (1.2%),Power BI (2.1%),Grafana (5.7%)



Top Linguagem por Área


area,Automação De Processos E Sistemas,Dados E Inteligência Artificial,Desenvolvimento De Software,Infraestrutura E Suporte,Projetos E Gerenciamento,Segurança Da Informação,Testes E Controle De Qualidade
Rank,,,,,,,
1,Python (16.8%),SQL (47.4%),Javascript (21.5%),SQL (8.8%),SQL (8.4%),Python (17.6%),SQL (7.9%)
2,SQL (10.7%),Javascript (5.2%),SQL (20.4%),Python (7.7%),Go (7.3%),Go (5.6%),Python (7.6%)
3,Javascript (5.4%),Go (4.1%),Java (18.6%),Go (4.4%),Python (4.4%),SQL (3.2%),Java (6.1%)
4,C++ (2.9%),Java (3.4%),Python (13.5%),Java (2.9%),C++ (2.9%),Java (2.4%),Javascript (5.0%)
5,Java (2.5%),C++ (3.3%),CSS (10.8%),Javascript (2.6%),Java (2.3%),C++ (1.9%),Go (3.8%)
6,Go (2.1%),Typescript (2.1%),HTML (10.6%),C++ (1.6%),Javascript (1.9%),C# (1.1%),Typescript (2.4%)
7,C# (1.4%),HTML (1.4%),Typescript (10.3%),C# (1.5%),C# (1.1%),GraphQL (0.8%),C++ (1.9%)
8,CSS (1.1%),C# (1.3%),C# (10.2%),Typescript (1.1%),HTML (0.9%),Kotlin (0.3%),C# (1.7%)
9,PHP (0.7%),CSS (0.8%),Go (8.1%),HTML (0.9%),GraphQL (0.9%),NaN,HTML (1.3%)



Top Conceito por Área


area,Automação De Processos E Sistemas,Dados E Inteligência Artificial,Desenvolvimento De Software,Infraestrutura E Suporte,Projetos E Gerenciamento,Segurança Da Informação,Testes E Controle De Qualidade
Rank,,,,,,,
1,Metodologias Ágeis (8.6%),Cloud (26.7%),API (29.9%),Cloud (22.1%),Metodologias Ágeis (20.0%),Cloud (42.6%),Metodologias Ágeis (15.5%)
2,API (8.2%),Machine Learning (21.2%),Metodologias Ágeis (21.4%),Metodologias Ágeis (12.8%),Scrum (9.6%),Data Privacy (19.1%),CI/CD (12.3%)
3,Lean (3.9%),Metodologias Ágeis (15.5%),Cloud (20.6%),CI/CD (8.4%),Cloud (9.0%),OWASP (13.3%),API (11.1%)
4,Cloud (2.9%),CI/CD (12.3%),CI/CD (19.9%),API (5.2%),Product Management (6.8%),CI/CD (13.0%),Scrum (9.1%)
5,Kanban (2.1%),API (11.8%),Microservices (13.2%),Data Privacy (3.5%),Kanban (5.8%),API (10.4%),Kanban (6.9%)
6,UI/UX Design (1.8%),NoSQL (6.0%),NoSQL (12.1%),Scrum (3.3%),Lean (3.2%),Metodologias Ágeis (8.8%),Cloud (6.2%)
7,NaN,Data Privacy (5.5%),Scrum (11.5%),Machine Learning (2.7%),CI/CD (3.0%),Pentest (6.4%),Lean (5.3%)
8,NaN,Scrum (4.2%),Kanban (8.0%),Microservices (2.4%),Data Privacy (2.6%),Microservices (2.7%),Data Privacy (2.7%)
9,NaN,Product Management (4.1%),Serverless (4.3%),Kanban (2.4%),Edge Computing (1.8%),Edge Computing (2.1%),Microservices (2.3%)


In [11]:
# ==========================================
# 11. Pós-graduação x Senioridade — formato executivo
# ==========================================

cols_pos = [c for c in df.columns if c in ['sk_posgraduacao', 'sk_mestrado', 'sk_doutorado', 'sk_mba']]

if cols_pos:
    df_pos = df.copy()
    df_pos['Tem_Pos'] = df_pos[cols_pos].sum(axis=1) > 0

    total_por_nivel = df_pos.groupby('nivel').size()
    qtd_pos = df_pos.groupby('nivel')['Tem_Pos'].sum()
    perc_pos = (qtd_pos / total_por_nivel) * 100

    res = pd.DataFrame({
        'Nivel': perc_pos.index,
        'Qtd': qtd_pos.values,
        'Perc': perc_pos.values
    }).sort_values('Perc', ascending=False)

    res['Rank'] = range(1, len(res) + 1)

    res['Texto'] = (
        res['Nivel']
        + ' — '
        + res['Perc'].round(1).astype(str)
        + '% ('
        + res['Qtd'].astype(int).astype(str)
        + ')'
    )

    pivot = res.set_index('Rank')[['Texto']]

    print("Porcentagem de vagas que pedem pós-graduação por nível")
    display(pivot)


Porcentagem de vagas que pedem pós-graduação por nível


,Texto
Rank,
1,Nível Liderança — 10.6% (166)
2,Nível Pleno / Sênior — 4.8% (351)
3,N/F — 4.5% (143)
4,Nível Júnior — 1.9% (64)


In [12]:
# ===============================
# 1. FUNÇÃO DE LIMPEZA DE TEXTO
# ===============================
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = unidecode(text.lower().strip())
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text


# ===============================
# 2. DEFINIÇÃO DOS NÍVEIS
# ===============================
levels_dict = {
    "junior": ["jr", "junior", "júnior", "entry", "entry level"],
    
    "estagio": [
        "estagio", "estagiario", "estagiaria",
        "intern", "internship"
    ],
    
    "aprendiz": [
        "aprendiz", "jovem aprendiz"
    ],
    
    "auxiliar": [
        "auxiliar", "assistente", "tecnico",
        "tecnica", "assistant", "operator"
    ],
    
    "trainee": [
        "trainee", "programa trainee"
    ]
}

# inclui o próprio nome como termo (evita duplicatas)
for level in levels_dict:
    levels_dict[level] = list(set(levels_dict[level] + [level]))


# ===============================
# 3. ESCOLHA A COLUNA DA VAGA
# ===============================
job_column = "Cargo"  # altere se necessário

if job_column not in df.columns:
    raise ValueError(f"Coluna '{job_column}' não encontrada no DataFrame.")

# cria versão limpa para busca
df["job_search"] = df[job_column].fillna("").apply(clean_text)


# ===============================
# 4. DETECTA TODOS OS NÍVEIS NO TEXTO
# ===============================
def detect_all_levels(text):
    found = []

    for level, terms in levels_dict.items():
        pattern = r'\b(' + '|'.join(map(re.escape, terms)) + r')\b'
        if re.search(pattern, text):
            found.append(level)

    return found if found else ["outros"]


# ===============================
# 5. FILTRA APENAS VAGAS JUNIOR
# (mesma lógica original)
# ===============================
df_junior = df[df["nivel"].str.contains("Junior|Júnior", case=False, na=False)].copy()

print("Total de vagas junior:", len(df_junior))


# ===============================
# 6. VERIFICA OUTROS NÍVEIS NAS JUNIOR
# ===============================
df_junior["niveis_encontrados"] = df_junior["job_search"].apply(detect_all_levels)

# explode para contagem
df_exploded = df_junior.explode("niveis_encontrados")

ranking = (
    df_exploded["niveis_encontrados"]
    .value_counts()
    .rename_axis("nivel")
    .reset_index(name="quantidade")
)

# adiciona porcentagem
total = ranking["quantidade"].sum()
ranking["porcentagem"] = (ranking["quantidade"] / total * 100).round(2).astype(str) + "%"

print("\nDistribuição dentro das vagas junior:")
display(ranking)

# ===============================
# 7. DISTRIBUIÇÃO SALARIAL POR LEVEL DENTRO DAS JUNIOR
# (apenas salários informados + ordem correta)
# ===============================

if "salario" not in df.columns:
    print("\n⚠️ Coluna 'salario' não encontrada.")
else:
    
    print("\nDistribuição salarial por level dentro das vagas junior (apenas salários informados):")

    df_junior_salary = df_junior.copy()
    df_junior_salary["salario"] = df.loc[df_junior_salary.index, "salario"]

    # remove salários não informados
    df_junior_salary = df_junior_salary[
        ~df_junior_salary["salario"].isin(["n/e", "n/f", "", None])
    ].dropna(subset=["salario"])

    # explode níveis detectados
    df_junior_salary = df_junior_salary.explode("niveis_encontrados")

    # cria matriz percentual
    salary_by_level = (
        pd.crosstab(
            df_junior_salary["niveis_encontrados"],
            df_junior_salary["salario"],
            normalize="index"
        ) * 100
    ).round(2)

    # ===============================
    # ORDENA AS FAIXAS SALARIAIS
    # ===============================
    def salary_sort_key(col):
        col = str(col).strip()

        # "|- 2" → 0
        if col.startswith("|-"):
            return 0

        # "30 |-" → valor alto
        if col.endswith("|-"):
            return int(col.split("|")[0].strip())

        # "2 |- 5" → usa número inicial
        return int(col.split("|")[0].strip())

    ordered_cols = sorted(salary_by_level.columns, key=salary_sort_key)
    salary_by_level = salary_by_level[ordered_cols]

    # adiciona %
    salary_by_level = salary_by_level.astype(str) + "%"

    display(salary_by_level)

Total de vagas junior: 3431

Distribuição dentro das vagas junior:


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13900/1682548606.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["job_search"] = df[job_column].fillna("").apply(clean_text)


,nivel,quantidade,porcentagem
0,auxiliar,1762,48.47%
1,junior,975,26.82%
2,estagio,805,22.15%
3,aprendiz,57,1.57%
4,trainee,36,0.99%



Distribuição salarial por level dentro das vagas junior (apenas salários informados):


salario
niveis_encontrados


In [13]:
from collections import Counter
import pandas as pd

stop_words = ['de','da','do','em','para','e','a','o','senior','pleno','junior','sr','pl','jr','i','ii','iii','analista','especialista']

def analyze_cargo_names(text_series):
    words = []
    for text in text_series:
        if isinstance(text, str):
            clean = clean_text_search(text)
            for w in clean.split():
                if w not in stop_words and len(w) > 2:
                    words.append(w)
    return Counter(words)

# ==========================================
# 13. NLP dos Cargos — formato executivo
# ==========================================

TOP_N = 3

res_nlp = []

for area, group in df[df['area'].notna()].groupby('area'):
    counter = analyze_cargo_names(group['Cargo'])
    total = sum(counter.values())

    for palavra, qtd in counter.items():
        res_nlp.append({
            'area': area,
            'Palavra': palavra,
            'Qtd': qtd,
            'Perc': (qtd / total) * 100 if total > 0 else 0
        })

df_nlp = pd.DataFrame(res_nlp)

df_nlp['Rank'] = (
    df_nlp.groupby('area')['Perc']
    .rank(method='dense', ascending=False)
    .astype(int)
)

df_nlp = df_nlp.sort_values(['area', 'Rank'])
df_nlp = df_nlp.groupby('area').head(TOP_N)

df_nlp['Texto'] = (
    df_nlp['Palavra']
    + ' ('
    + df_nlp['Perc'].round(1).astype(str)
    + '%)'
    + ' (' + df_nlp['Rank'].astype(str) + 'º)'
)

pivot = df_nlp.pivot_table(index='Rank', columns='area', values='Texto', aggfunc='first').sort_index()

print("\nNLP dos Cargos por Área — Termos Dominantes")
display(pivot)



NLP dos Cargos por Área — Termos Dominantes


area,Automação De Processos E Sistemas,Dados E Inteligência Artificial,Desenvolvimento De Software,Infraestrutura E Suporte,Projetos E Gerenciamento,Segurança Da Informação,Testes E Controle De Qualidade
Rank,,,,,,,
1,automacao (20.5%) (1º),dados (10.3%) (1º),desenvolvedor (5.5%) (1º),suporte (10.0%) (1º),projetos (5.0%) (1º),seguranca (12.5%) (1º),qualidade (18.1%) (1º)
2,tecnico (6.3%) (2º),data (4.6%) (2º),software (3.9%) (2º),tecnico (5.5%) (2º),tecnologia (4.6%) (2º),informacao (10.4%) (2º),engineer (2.5%) (2º)
3,automation (3.1%) (3º),inteligencia (4.0%) (3º),desenvolvimento (3.8%) (3º),infraestrutura (4.3%) (3º),sap (3.5%) (3º),security (7.9%) (3º),quality (2.2%) (3º)


In [14]:
# ==========================================
# 14/15. Top Skills Técnicas — formato executivo
# ==========================================

TOP_N = 10

df_meta_tech = df_meta[df_meta['Tipo'].isin(['hard skill', 'programação'])]
cols_tech = df_meta_tech['col_name'].tolist()

# ----------
# GERAL
# ----------

total_vagas = len(df)

soma = df[cols_tech].sum().sort_values(ascending=False).head(TOP_N)

res = pd.DataFrame({
    'col_name': soma.index,
    'Qtd': soma.values
}).merge(df_meta_tech[['col_name', 'Skill Real']], on='col_name')

res['Perc'] = (res['Qtd'] / total_vagas) * 100
res['Rank'] = range(1, len(res) + 1)

res['Texto'] = (
    res['Skill Real']
    + ' — '
    + res['Perc'].round(1).astype(str)
    + '% ('
    + res['Qtd'].astype(int).astype(str)
    + ')'
)

pivot_geral = res.set_index('Rank')[['Texto']]

print("\nTop Skills Técnicas — Geral")
display(pivot_geral)


# ----------
# POR ÁREA
# ----------

total_por_area = df.groupby('area').size()

res_area = df.groupby('area')[cols_tech].sum().reset_index()
res_area = res_area.melt(id_vars='area', var_name='col_name', value_name='Qtd')
res_area = res_area.merge(df_meta_tech[['col_name', 'Skill Real']], on='col_name')

res_area['Perc'] = res_area.apply(
    lambda x: (x['Qtd'] / total_por_area[x['area']]) * 100,
    axis=1
)

res_area['Rank'] = (
    res_area.groupby('area')['Perc']
    .rank(method='dense', ascending=False)
    .astype(int)
)

res_area = res_area.sort_values(['area', 'Rank'])
res_area = res_area.groupby('area').head(5)

res_area['Texto'] = res_area['Skill Real'] + ' (' + res_area['Perc'].round(1).astype(str) + '%)'

pivot_area = res_area.pivot_table(index='Rank', columns='area', values='Texto', aggfunc='first').sort_index()

print("\nTop Skills Técnicas — Por Área")
display(pivot_area)



Top Skills Técnicas — Geral


,Texto
Rank,
1,Cloud — 18.5% (2877)
2,SQL — 18.2% (2825)
3,Metodologias Ágeis — 17.4% (2704)
4,Python — 15.4% (2393)
5,API — 14.5% (2256)
6,AWS — 13.8% (2137)
7,Excel — 13.1% (2033)
8,CI/CD — 11.7% (1812)
9,Git — 10.6% (1645)



Top Skills Técnicas — Por Área


area,Automação De Processos E Sistemas,Dados E Inteligência Artificial,Desenvolvimento De Software,Infraestrutura E Suporte,Projetos E Gerenciamento,Segurança Da Informação,Testes E Controle De Qualidade
Rank,,,,,,,
1,Excel (17.9%),SQL (47.4%),API (29.9%),Windows (22.5%),Metodologias Ágeis (20.0%),Cloud (42.6%),Excel (19.2%)
2,Python (16.8%),Power BI (30.8%),Git (22.9%),Cloud (22.1%),SAP (17.6%),Splunk (29.8%),Office 365 (18.6%)
3,SQL (10.7%),Cloud (26.7%),Javascript (21.5%),Linux (16.0%),Excel (14.5%),AWS (28.5%),Metodologias Ágeis (15.5%)
4,Office 365 (10.0%),Excel (24.1%),Metodologias Ágeis (21.4%),AWS (13.9%),Scrum (9.6%),Azure (26.3%),CI/CD (12.3%)
5,Metodologias Ágeis (8.6%),NaN,Cloud (20.6%),Office 365 (13.7%),Office 365 (9.4%),GCP (20.2%),API (11.1%)


---
## LLMs Citadas nas Vagas

Detecta menções explícitas a modelos de linguagem específicos (Claude, ChatGPT, Cursor, Gemini, Deepseek, Kimi, Grok, Ollama) nas descrições das vagas.

In [15]:
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

import pandas as pd
import re

# Carrega Skills do scp_02 (coluna não presente no scp_03)
df2 = pd.read_csv('data/scp_02.csv')
df['Skills'] = df2['Skills'].values
df['Skills_Search'] = df['Skills'].fillna('').apply(clean_text_search)

LLMS = {
    'Claude':   r'\bclaude\b',
    'ChatGPT':  r'\b(chatgpt|chat gpt)\b',
    'Cursor':   r'\bcursor\b',
    'Gemini':   r'\bgemini\b',
    'Deepseek': r'\b(deepseek|deep seek)\b',
    'Kimi':     r'\bkimi\b',
    'Grok':     r'\bgrok\b',
    'Ollama':   r'\bollama\b',
}

llm_cols = []
for llm, pat in LLMS.items():
    col = f'llm_{llm.lower()}'
    df[col] = df['Skills_Search'].str.contains(pat, regex=True, na=False).astype(int)
    llm_cols.append(col)

df['qualquer_llm'] = df[llm_cols].max(axis=1)

total = len(df)
print('=== LLMs detectadas ===')
for llm, col in zip(LLMS, llm_cols):
    n = df[col].sum()
    print(f'  {llm:10s}: {n:4d} vagas  ({n/total*100:.1f}%)')
print(f'\n  Qualquer LLM: {df["qualquer_llm"].sum()} vagas ({df["qualquer_llm"].sum()/total*100:.1f}%)')


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13900/2887252589.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df['Skills_Search'].str.contains(pat, regex=True, na=False).astype(int)


/var/folders/08/3dcsxbrd36z11lbnx_70fhb40000gn/T/ipykernel_13900/2887252589.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df['Skills_Search'].str.contains(pat, regex=True, na=False).astype(int)


=== LLMs detectadas ===
  Claude    :  392 vagas  (2.5%)
  ChatGPT   :  155 vagas  (1.0%)
  Cursor    :  163 vagas  (1.1%)
  Gemini    :  154 vagas  (1.0%)
  Deepseek  :    2 vagas  (0.0%)
  Kimi      :    2 vagas  (0.0%)
  Grok      :    6 vagas  (0.0%)
  Ollama    :    0 vagas  (0.0%)

  Qualquer LLM: 528 vagas (3.4%)


In [16]:
# --- Ranking geral ---
resumo_rows = []
for llm, col in zip(LLMS, llm_cols):
    n = int(df[col].sum())
    resumo_rows.append({'LLM': llm, 'Vagas': n, '% do total': round(n/len(df)*100, 1)})
df_resumo_llm = pd.DataFrame(resumo_rows).sort_values('Vagas', ascending=False)
print('--- Ranking geral ---')
display(df_resumo_llm)

# --- Por área ---
areas = df['area'].value_counts().index.tolist()
rows_area = []
for llm, col in zip(LLMS, llm_cols):
    row = {'LLM': llm}
    for area in areas:
        mask = df['area'] == area
        total_area = mask.sum()
        n = int(df.loc[mask, col].sum())
        row[area] = f'{n} ({n/total_area*100:.1f}%)' if total_area else '0'
    rows_area.append(row)
df_llm_area = pd.DataFrame(rows_area).set_index('LLM')
print('\n--- LLMs por área ---')
display(df_llm_area)


--- Ranking geral ---


,LLM,Vagas,% do total
0,Claude,392,2.5
2,Cursor,163,1.1
1,ChatGPT,155,1.0
3,Gemini,154,1.0
6,Grok,6,0.0
4,Deepseek,2,0.0
5,Kimi,2,0.0
7,Ollama,0,0.0



--- LLMs por área ---


,Desenvolvimento De Software,Projetos E Gerenciamento,Infraestrutura E Suporte,Dados E Inteligência Artificial,Testes E Controle De Qualidade,Segurança Da Informação,Automação De Processos E Sistemas
LLM,,,,,,,
Claude,152 (3.2%),56 (1.7%),27 (0.8%),126 (5.1%),18 (1.6%),5 (1.3%),8 (2.9%)
ChatGPT,48 (1.0%),22 (0.7%),18 (0.6%),50 (2.0%),9 (0.8%),0 (0.0%),8 (2.9%)
Cursor,98 (2.1%),23 (0.7%),10 (0.3%),25 (1.0%),4 (0.4%),2 (0.5%),1 (0.4%)
Gemini,31 (0.7%),16 (0.5%),4 (0.1%),86 (3.5%),5 (0.5%),3 (0.8%),9 (3.2%)
Deepseek,1 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
Kimi,1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
Grok,5 (0.1%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
Ollama,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


In [17]:
# --- Por nível de senioridade ---
rows_nivel = []
for llm, col in zip(LLMS, llm_cols):
    row = {'LLM': llm}
    for nivel in df['nivel'].unique():
        mask = df['nivel'] == nivel
        total_nivel = mask.sum()
        n = int(df.loc[mask, col].sum())
        row[nivel] = f'{n} ({n/total_nivel*100:.1f}%)' if total_nivel else '0'
    rows_nivel.append(row)
df_llm_nivel = pd.DataFrame(rows_nivel).set_index('LLM')
print('--- LLMs por nível de senioridade ---')
display(df_llm_nivel)


--- LLMs por nível de senioridade ---


,N/F,Nível Liderança,Nível Júnior,Nível Pleno / Sênior
LLM,,,,
Claude,68 (2.2%),48 (3.1%),36 (1.0%),240 (3.3%)
ChatGPT,14 (0.4%),18 (1.1%),35 (1.0%),88 (1.2%)
Cursor,48 (1.5%),27 (1.7%),10 (0.3%),78 (1.1%)
Gemini,14 (0.4%),9 (0.6%),25 (0.7%),106 (1.4%)
Deepseek,0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%)
Kimi,0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%)
Grok,2 (0.1%),0 (0.0%),0 (0.0%),4 (0.1%)
Ollama,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


In [18]:
# --- Co-ocorrências ---
llm_names = list(LLMS.keys())
co_rows = []
for i, a in enumerate(llm_names):
    for b in llm_names[i+1:]:
        col_a, col_b = f'llm_{a.lower()}', f'llm_{b.lower()}'
        n = int((df[col_a] & df[col_b]).sum())
        if n > 0:
            co_rows.append({
                'LLM A': a, 'LLM B': b, 'Vagas com ambos': n,
                '% de A': round(n / max(df[col_a].sum(), 1) * 100, 1),
                '% de B': round(n / max(df[col_b].sum(), 1) * 100, 1),
            })
df_co = pd.DataFrame(co_rows).sort_values('Vagas com ambos', ascending=False)
print('--- Co-ocorrências entre LLMs ---')
display(df_co)

# --- LLMs vs flag IA ---
print('\n--- Vagas com LLM específica vs flag IA=1 ---')
for llm, col in zip(LLMS, llm_cols):
    llm_total = int(df[col].sum())
    llm_com_ia = int(((df[col] == 1) & (df['IA'] == 1)).sum())
    if llm_total > 0:
        print(f'  {llm:10s}: {llm_total} vagas | {llm_com_ia} com IA=1 ({llm_com_ia/llm_total*100:.0f}%)')


--- Co-ocorrências entre LLMs ---


,LLM A,LLM B,Vagas com ambos,% de A,% de B
1,Claude,Cursor,120,30.6,73.6
0,Claude,ChatGPT,104,26.5,67.1
2,Claude,Gemini,90,23.0,58.4
7,ChatGPT,Gemini,58,37.4,37.7
6,ChatGPT,Cursor,33,21.3,20.2
9,Cursor,Gemini,15,9.2,9.7
5,Claude,Grok,5,1.3,83.3
10,Cursor,Grok,4,2.5,66.7
8,ChatGPT,Grok,4,2.6,66.7
4,Claude,Kimi,2,0.5,100.0



--- Vagas com LLM específica vs flag IA=1 ---
  Claude    : 392 vagas | 392 com IA=1 (100%)
  ChatGPT   : 155 vagas | 155 com IA=1 (100%)
  Cursor    : 163 vagas | 162 com IA=1 (99%)
  Gemini    : 154 vagas | 154 com IA=1 (100%)
  Deepseek  : 2 vagas | 2 com IA=1 (100%)
  Kimi      : 2 vagas | 2 com IA=1 (100%)
  Grok      : 6 vagas | 6 com IA=1 (100%)


---
## Investigação: Queda de Vagas com Machine Learning (2025 → 2026)

Compara a presença do termo *Machine Learning* nas descrições de vagas entre os dois levantamentos e decompõe a queda em componente de **volume** (menos vagas tech no geral) e componente **estrutural** (ML citado proporcionalmente menos).

In [19]:
# ==========================================
# SETUP — carrega dados de 2025 e 2026
# ==========================================

PATH_25 = '../../../../webscrapping-2025/Códigos Webscrapping Paulo/data/'
PATH_26 = 'data/'

df25 = pd.read_csv(PATH_25 + 'scp_02.csv')
df26 = pd.read_csv(PATH_26 + 'scp_02.csv')
sk25 = pd.read_csv(PATH_25 + 'scp_03.csv')
sk26 = pd.read_csv(PATH_26 + 'scp_03.csv')

n25, n26 = len(df25), len(df26)
ml25 = int(sk25['sk_machine_learning'].sum())
ml26 = int(sk26['sk_machine_learning'].sum())
ia25 = int(sk25['IA'].sum())
ia26 = int(sk26['IA'].sum())

print(f'Vagas totais   —  2025: {n25:,}  |  2026: {n26:,}  |  Δ: {n26-n25:+,} ({(n26-n25)/n25*100:.1f}%)')
print(f'Vagas com ML   —  2025: {ml25:,} ({ml25/n25*100:.1f}%)  |  2026: {ml26:,} ({ml26/n26*100:.1f}%)  |  Δ: {ml26-ml25:+,}')
print(f'Vagas com IA   —  2025: {ia25:,} ({ia25/n25*100:.1f}%)  |  2026: {ia26:,} ({ia26/n26*100:.1f}%)  |  Δ: {ia26-ia25:+,}')

Vagas totais   —  2025: 19,293  |  2026: 15,514  |  Δ: -3,779 (-19.6%)
Vagas com ML   —  2025: 1,343 (7.0%)  |  2026: 802 (5.2%)  |  Δ: -541
Vagas com IA   —  2025: 1,523 (7.9%)  |  2026: 3,837 (24.7%)  |  Δ: +2,314


In [20]:
# ==========================================
# 1. DECOMPOSIÇÃO DA QUEDA DE ML
#    volume total  ×  queda estrutural
# ==========================================

ml_esperado = n26 * (ml25 / n25)
queda_volume     = ml_esperado - ml25          # negativo: menos vagas no total
queda_estrutural = ml26 - ml_esperado          # negativo: ML menos citado

queda_total = ml26 - ml25

print('=== Decomposição da queda de Machine Learning ===\n')
print(f'  Queda total de vagas ML          : {queda_total:+.0f} vagas  (-{abs(queda_total)/ml25*100:.1f}%)')
print(f'  └─ Explicada pela queda de volume: {queda_volume:+.0f} vagas  ({queda_volume/queda_total*100:.1f}%)')
print(f'  └─ Queda estrutural (ML menos citado): {queda_estrutural:+.0f} vagas  ({queda_estrutural/queda_total*100:.1f}%)')
print()
print('  Se a % de ML de 2025 (7,0%) se mantivesse no volume de 2026:')
print(f'  → seriam esperadas {ml_esperado:.0f} vagas ML; observamos apenas {ml26}.')

# ==========================================
# 2. ML POR ÁREA
# ==========================================
print('\n=== ML por Área ===\n')

areas = sorted(df26['Area'].dropna().unique())
rows_area = []
for area in areas:
    idx25 = sk25.index[df25['Area'] == area]
    idx26 = sk26.index[df26['Area'] == area]
    n25_a, n26_a = len(idx25), len(idx26)
    ml25_a = int(sk25.loc[idx25, 'sk_machine_learning'].sum()) if n25_a else 0
    ml26_a = int(sk26.loc[idx26, 'sk_machine_learning'].sum()) if n26_a else 0
    p25_a = ml25_a / n25_a * 100 if n25_a else 0
    p26_a = ml26_a / n26_a * 100 if n26_a else 0
    rows_area.append({
        'Área': area,
        'ML 2025 (n)': ml25_a, 'ML 2025 (%)': round(p25_a, 1),
        'ML 2026 (n)': ml26_a, 'ML 2026 (%)': round(p26_a, 1),
        'Δ (n)': ml26_a - ml25_a,  'Δ pp': round(p26_a - p25_a, 1),
    })

df_ml_area = pd.DataFrame(rows_area).sort_values('Δ (n)')
display(df_ml_area.set_index('Área'))

ml_ia_25 = int(sk25.loc[sk25.index[df25['Area'] == 'Dados e Inteligência Artificial'], 'sk_machine_learning'].sum())
ml_ia_26 = int(sk26.loc[sk26.index[df26['Area'] == 'Dados e Inteligência Artificial'], 'sk_machine_learning'].sum())
pct_explicado = (ml_ia_25 - ml_ia_26) / abs(queda_total) * 100
print(f'\n  A área "Dados e IA" concentra {pct_explicado:.1f}% da perda absoluta de vagas ML.')

# ==========================================
# 3. ML POR NÍVEL DE SENIORIDADE
# ==========================================
print('\n=== ML por Nível de Senioridade ===\n')

niveis = ['Nível Júnior', 'Nível Pleno / Sênior', 'Nível Liderança', 'n/f']
rows_niv = []
for nivel in niveis:
    idx25 = sk25.index[df25['Nivel'] == nivel]
    idx26 = sk26.index[df26['Nivel'] == nivel]
    n25_n, n26_n = len(idx25), len(idx26)
    ml25_n = int(sk25.loc[idx25, 'sk_machine_learning'].sum()) if n25_n else 0
    ml26_n = int(sk26.loc[idx26, 'sk_machine_learning'].sum()) if n26_n else 0
    p25_n = ml25_n / n25_n * 100 if n25_n else 0
    p26_n = ml26_n / n26_n * 100 if n26_n else 0
    rows_niv.append({
        'Nível': nivel,
        'ML 2025 (n)': ml25_n, 'ML 2025 (%)': round(p25_n, 1),
        'ML 2026 (n)': ml26_n, 'ML 2026 (%)': round(p26_n, 1),
        'Δ pp': round(p26_n - p25_n, 1),
    })

df_ml_nivel = pd.DataFrame(rows_niv)
display(df_ml_nivel.set_index('Nível'))

=== Decomposição da queda de Machine Learning ===

  Queda total de vagas ML          : -541 vagas  (-40.3%)
  └─ Explicada pela queda de volume: -263 vagas  (48.6%)
  └─ Queda estrutural (ML menos citado): -278 vagas  (51.4%)

  Se a % de ML de 2025 (7,0%) se mantivesse no volume de 2026:
  → seriam esperadas 1080 vagas ML; observamos apenas 802.

=== ML por Área ===



,ML 2025 (n),ML 2025 (%),ML 2026 (n),ML 2026 (%),Δ (n),Δ pp
Área,,,,,,
Dados e Inteligência Artificial,930,28.3,520,21.2,-410,-7.1
Desenvolvimento de Software,212,3.0,113,2.4,-99,-0.7
Projetos e Gerenciamento,84,3.1,57,1.7,-27,-1.4
Automação de Processos e Sistemas,20,3.7,3,1.1,-17,-2.7
Testes e Controle de Qualidade,27,2.6,17,1.5,-10,-1.1
Segurança da Informação,8,1.7,7,1.9,-1,0.2
Infraestrutura e Suporte,62,1.5,85,2.7,23,1.2



  A área "Dados e IA" concentra 75.8% da perda absoluta de vagas ML.

=== ML por Nível de Senioridade ===



,ML 2025 (n),ML 2025 (%),ML 2026 (n),ML 2026 (%),Δ pp
Nível,,,,,
Nível Júnior,148,2.8,66,1.9,-0.9
Nível Pleno / Sênior,813,8.4,460,6.3,-2.2
Nível Liderança,143,9.3,101,6.4,-2.9
n/f,239,8.3,175,5.6,-2.7


In [21]:
# ==========================================
# 4. HIPÓTESE: ML ABSORVIDO PELA LINGUAGEM "IA"
# ==========================================

ml_com_ia_25 = int(((sk25['IA'] == 1) & (sk25['sk_machine_learning'] == 1)).sum())
ml_com_ia_26 = int(((sk26['IA'] == 1) & (sk26['sk_machine_learning'] == 1)).sum())
ml_sem_ia_25 = int(((sk25['IA'] == 0) & (sk25['sk_machine_learning'] == 1)).sum())
ml_sem_ia_26 = int(((sk26['IA'] == 0) & (sk26['sk_machine_learning'] == 1)).sum())
ia_sem_ml_25 = int(((sk25['IA'] == 1) & (sk25['sk_machine_learning'] == 0)).sum())
ia_sem_ml_26 = int(((sk26['IA'] == 1) & (sk26['sk_machine_learning'] == 0)).sum())

print('=== Relação entre flag IA e skill Machine Learning ===\n')
comp_ia_ml = pd.DataFrame({
    'Métrica': [
        'Vagas com ML + flag IA',
        'Vagas com ML sem flag IA',
        'Vagas com IA mas sem ML',
    ],
    '2025 (n)': [ml_com_ia_25, ml_sem_ia_25, ia_sem_ml_25],
    '2025 (% das vagas ML)': [
        f'{ml_com_ia_25/ml25*100:.1f}%',
        f'{ml_sem_ia_25/ml25*100:.1f}%',
        '—',
    ],
    '2026 (n)': [ml_com_ia_26, ml_sem_ia_26, ia_sem_ml_26],
    '2026 (% das vagas ML)': [
        f'{ml_com_ia_26/ml26*100:.1f}%',
        f'{ml_sem_ia_26/ml26*100:.1f}%',
        '—',
    ],
    'Δ (n)': [
        ml_com_ia_26 - ml_com_ia_25,
        ml_sem_ia_26 - ml_sem_ia_25,
        ia_sem_ml_26 - ia_sem_ml_25,
    ],
})
display(comp_ia_ml.set_index('Métrica'))

print()
print('Interpretação:')
print('  Em 2025, 35,4% das vagas com ML não eram classificadas como IA.')
print('  Em 2026, apenas 1,5% das vagas com ML ficam fora da flag IA.')
print('  Ao mesmo tempo, as vagas IA sem ML saltaram de 656 → 3.047.')
print('  Sinal claro de que "Machine Learning" virou termo específico dentro do')
print('  guarda-chuva "IA" — as empresas passaram a usar IA como rótulo genérico.')

# ==========================================
# 5. SHIFT DE SKILLS NA ÁREA DADOS E IA
# ==========================================
print('\n=== Mudança de perfil de skills — Área: Dados e Inteligência Artificial ===\n')

area = 'Dados e Inteligência Artificial'
idx25_d = sk25.index[df25['Area'] == area]
idx26_d = sk26.index[df26['Area'] == area]
n25_d, n26_d = len(idx25_d), len(idx26_d)

SK_COLS = [c for c in sk26.columns if c.startswith('sk_')]
SK_COMUNS = [c for c in SK_COLS if c in sk25.columns]

p25_d = (sk25.loc[idx25_d, SK_COMUNS].sum() / n25_d * 100)
p26_d = (sk26.loc[idx26_d, SK_COMUNS].sum() / n26_d * 100)
delta_d = (p26_d - p25_d).round(1)

top_media = ((p25_d + p26_d) / 2).sort_values(ascending=False).head(20)

df_crit_sk = pd.read_excel('data/criterios.xlsx', sheet_name='skills')
def normalize_col(text):
    import re
    from unidecode import unidecode
    t = str(text).lower().strip()
    for k, v in [('c#','csharp'),('c++','cplusplus'),('.net','dotnet'),('node.js','nodejs')]:
        t = t.replace(k, v)
    t = unidecode(t)
    t = re.sub(r'[^a-z0-9]+', '_', t).strip('_')
    return f'sk_{t}'
df_crit_sk['col_name'] = df_crit_sk['Skill'].apply(normalize_col)
skill_label = df_crit_sk.set_index('col_name')['Skill']

rows_shift = []
for c in top_media.index:
    delta = delta_d[c]
    rows_shift.append({
        'Skill': skill_label.get(c, c),
        '2025 (%)': round(p25_d[c], 1),
        '2026 (%)': round(p26_d[c], 1),
        'Δ pp': round(delta, 1),
        'Tendência': '↑' if delta > 0 else ('↓' if delta < 0 else '='),
    })

df_shift = pd.DataFrame(rows_shift, index=range(1, len(rows_shift)+1))
df_shift.index.name = 'Rank'
print(f'Área: Dados e IA  |  2025: {n25_d:,} vagas  →  2026: {n26_d:,} vagas  (queda de {(n26_d-n25_d)/n25_d*100:.1f}%)\n')
display(df_shift)

=== Relação entre flag IA e skill Machine Learning ===



,2025 (n),2025 (% das vagas ML),2026 (n),2026 (% das vagas ML),Δ (n)
Métrica,,,,,
Vagas com ML + flag IA,867,64.6%,790,98.5%,-77
Vagas com ML sem flag IA,476,35.4%,12,1.5%,-464
Vagas com IA mas sem ML,656,—,3047,—,2391



Interpretação:
  Em 2025, 35,4% das vagas com ML não eram classificadas como IA.
  Em 2026, apenas 1,5% das vagas com ML ficam fora da flag IA.
  Ao mesmo tempo, as vagas IA sem ML saltaram de 656 → 3.047.
  Sinal claro de que "Machine Learning" virou termo específico dentro do
  guarda-chuva "IA" — as empresas passaram a usar IA como rótulo genérico.

=== Mudança de perfil de skills — Área: Dados e Inteligência Artificial ===

Área: Dados e IA  |  2025: 3,291 vagas  →  2026: 2,454 vagas  (queda de -25.4%)



,Skill,2025 (%),2026 (%),Δ pp,Tendência
Rank,,,,,
1,Python,50.7,47.4,-3.3,↓
2,SQL,50.1,47.4,-2.8,↓
3,Comunicação,39.5,44.8,5.3,↑
4,Engenharia,37.5,46.5,9.0,↑
5,Criatividade,37.5,33.9,-3.6,↓
6,Organização,27.7,35.2,7.6,↑
7,Power BI,31.0,30.8,-0.2,↓
8,Pensamento analítico,30.8,26.7,-4.0,↓
9,Cloud,23.6,26.7,3.1,↑


---
## Títulos de Cargo Mais Frequentes por Área

Normaliza os títulos removendo marcadores de senioridade (Júnior, Sênior, Pleno, Jr, Sr, Trainee, Estagiário)
e conta os títulos-base mais comuns dentro de cada área.

In [22]:
import re
import pandas as pd

df_cargos = pd.read_csv('data/scp_02.csv')

# ── 1. Sufixos de modalidade/região ao final do título ────────────────────
# Ex: '- remote, latin america', '| LATAM | - remote work'
_MODALIDADE_PAT = re.compile(
    r'\s*[|]\s*(?:latam|remote|brasil|brazil).*$'
    r'|\s+[-–]\s*(?:remote(?:\s+work)?(?:[\s,]*(?:latin\s+america|latam|brasil|brazil))?'
    r'|latam|latin\s+america|100\s*%\s*remoto?).*$',
    re.I,
)

# ── 2. Nome de empresa entre parênteses ───────────────────────────────────
# Remove (act digital), (unique solutions), (success people) etc.
# Mantém siglas em maiúsculo: (QA), (TI), (BI), (L2/L3)
_EMPRESA_PAT = re.compile(
    r'\([^)]*[a-záàâãéèêíìïóòôõúùûç][^)]*\)',
    re.UNICODE,
)

# ── 3. Marcadores de senioridade ─────────────────────────────────────────
_SENIOR_PAT = re.compile(
    r'(?<![\w])(?:'
    r'j[uú]nior|s[eê]nior|pleno|estagiári[ao]|trainee|intern(?:ship)?'
    r'|jr\.?|sr\.?'
    r'|n[íi]vel\s+[ivx]+'
    r'|\b(?:ii|iii|iv|vi|vii|viii)\b'
    r')(?![\w])',
    re.I | re.UNICODE,
)

def normaliza_cargo(cargo):
    if not isinstance(cargo, str):
        return ''
    t = cargo
    t = _MODALIDADE_PAT.sub('', t)           # remove sufixo remoto/região
    t = _EMPRESA_PAT.sub('', t)              # remove (nome da empresa)
    t = _SENIOR_PAT.sub(' ', t)              # remove marcador de senioridade
    t = re.sub(r'full\s*stack', 'full stack', t, flags=re.I)
    t = re.sub(r'front[\s\-]*end', 'front-end', t, flags=re.I)
    t = re.sub(r'back[\s\-]*end', 'back-end', t, flags=re.I)
    t = re.sub(r'\s{2,}', ' ', t).strip(' -/,.|')
    return t.strip()

df_cargos['cargo_norm'] = df_cargos['Cargo'].apply(normaliza_cargo)

AREAS_VALIDAS = [
    a for a in df_cargos['Area'].value_counts().index
    if a not in ('n/f', 'n/e')
]

TOP_N = 15

for area in AREAS_VALIDAS:
    sub = df_cargos[df_cargos['Area'] == area]
    total = len(sub)
    top = sub['cargo_norm'].value_counts().head(TOP_N)
    print(f"\n{'='*58}")
    print(f"  {area.upper()}  ({total} vagas)")
    print(f"{'='*58}")
    for cargo, cnt in top.items():
        pct = cnt / total * 100
        print(f"  {cnt:4d}  ({pct:4.1f}%)  {cargo}")



  DESENVOLVIMENTO DE SOFTWARE  (4758 vagas)
   127  ( 2.7%)  desenvolvedor full stack
    95  ( 2.0%)  analista de sistemas
    69  ( 1.5%)  desenvolvedor back-end
    57  ( 1.2%)  software engineer
    43  ( 0.9%)  sdr - sales development representative
    39  ( 0.8%)  sales development representative
    37  ( 0.8%)  desenvolvedor front-end
    27  ( 0.6%)  desenvolvedor java
    27  ( 0.6%)  desenvolvedor . net
    27  ( 0.6%)  engenheiro de software
    21  ( 0.4%)  back-end engineer
    17  ( 0.4%)  analista de desenvolvimento de sistemas
    15  ( 0.3%)  desenvolvedor de software
    14  ( 0.3%)  front-end engineer
    14  ( 0.3%)  full stack developer

  PROJETOS E GERENCIAMENTO  (3349 vagas)
    63  ( 1.9%)  analista de projetos
    25  ( 0.7%)  business development representative
    23  ( 0.7%)  consultor sap sd
    20  ( 0.6%)  gerente de projetos
    19  ( 0.6%)  analista de produto
    18  ( 0.5%)  consultor sap abap
    18  ( 0.5%)  consultor sap btp
    16  ( 0.5%)  co